# Rediscover the Higgs Boson — Guided Practice Notebook

This notebook follows the same analysis path as **HyyAnalysis.ipynb** —
rediscovering the Higgs boson in the $H\rightarrow\gamma\gamma$ decay channel using
ATLAS Open Data — but breaks the journey into many smaller **tasks** that you
should complete yourself.

> **How to use this notebook**
>
> Every task is clearly marked with a coloured banner:
>
> <div class="alert alert-block alert-warning">
> <b>>>></b> TASK — short description
> </div>
>
> The solution code is provided right below each task so that you can check your
> work.  When doing the exercise yourself, clear the solution cells first and
> write your own implementation.
>
> Cells that are **not** part of a task (setup, data access, plotting utilities)
> should simply be run as-is.

**Learning objectives**
1. Understand and implement analysis selections (cuts) on ATLAS open data.
2. Quantify the effect of each cut with a cut-flow table.
3. Study the di-photon invariant mass spectrum before and after selections.
4. Fit the background with a polynomial and the signal with a Gaussian.
5. Evaluate fit quality with residuals and reduced $\chi^2$.
6. Estimate the size of the Higgs excess in events and as a simple significance.
7. Reason about setting a limit with and without a signal.

---

## Table of Contents
1. [Setup & Data Access](#setup)
2. [Task 1 — Explore the TTree](#task1)
3. [Task 2 — Define the cut functions](#task2)
4. [Task 3 — Apply cuts to a single batch](#task3)
5. [Task 4 — Cut-flow table](#task4)
6. [Task 5 — Invariant mass of a single event](#task5)
7. [Task 6 — Histogram the data after cuts](#task6)
8. [Task 7 — Background-only fit](#task7)
9. [Task 8 — Signal+background fit](#task8)
10. [Task 9 — Residuals and reduced $\chi^2$](#task9)
11. [Task 10 — Signal yield and excess estimate](#task10)
12. [Task 11 — Sideband cross-check](#task11)
13. [Task 12 — Limit-setting reasoning](#task12)
14. [Task 13 — Optional: vary a cut](#task13)


<a id='setup'></a>
## 1. Setup & Data Access

We import the usual python packages and initialise `atlasopenmagic` to access the
ATLAS Open Data portal.

The conda environment **cern_intern_practice** contains all needed packages.

In [ ]:
# Core packages
import uproot                 # reading ROOT files
import time                   # timing
import math
import awkward as ak          # jagged arrays
import numpy as np            # numerics
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator, AutoMinorLocator
from lmfit.models import PolynomialModel, GaussianModel
import vector                 # 4-vector math

# ATLAS Open Data access
import atlasopenmagic as atom
atom.set_release('2025e-13tev-beta')

print("Setup complete.")


### Analysis parameters

* **luminosity** — the total integrated luminosity of the data sample (36.1 fb$^{-1}$).
* **fraction** — fraction of events to process (reduce for speed; set to 1 for full data).


In [ ]:
luminosity = 36.1  # fb^-1
fraction   = 1.0    # 1.0 = all data; 0.5 = half (faster)

# Skim: pre-selected events with >= 2 photons
skim = "GamGam"

# Retrieve data file URLs (cached locally)
files_list = atom.get_urls('data', skim, protocol='https', cache=True)
print(files_list)
print(f"Number of data files: {len(files_list)}")


### Important variables

For the $H\rightarrow\gamma\gamma$ analysis we only need a handful of branches.
The `photon_*` branches store per-photon quantities as variable-length lists
(two entries per event in this skim, ordered by descending $p_T$).


In [ ]:
variables = [
    "photon_pt",            # transverse momentum [GeV]
    "photon_eta",           # pseudorapidity
    "photon_phi",           # azimuthal angle
    "photon_e",             # energy [GeV]
    "photon_isTightID",     # tight identification flag
    "photon_ptcone20",      # track-based isolation variable
]

# For MC we also need weight information
mc_weight_variables = ["mcWeight", "xsec", "kfac", "filteff", "sum_of_weights"]


### Quick look at a single file

Next, let's open the `data15_periodD` file to examine its contents. The file, known as a `tree`, contains multiple entries, each representing an event. For each event, a dictionary stores all relevant information with keys, such as the event number (`eventNumber`), the photon transverse momentum (`photon_pt`), and more.

Details on the variables in the dictionary can be viewed [here](https://cds.cern.ch/record/2707171/files/ANA-OTRC-2019-01-PUB-updated.pdf) in Appendix A.

More information on trees can be viewed [here](https://uproot.readthedocs.io/en/latest/uproot.behaviors.TTree.TTree.html) and [here](https://hsf-training.github.io/hsf-training-uproot-webpage/03-trees/index.html).

In [ ]:
# Accessing the file from the online database (":analysis" opens the tree in a desired manner)
tree = uproot.open(files_list[0] + ":analysis")
print("Entries in file 0:", tree.num_entries)
print("Keys (first 12):", tree.keys()[:12])


<a id='task1'></a>
## 2. Task 1 — Explore the TTree

<div class="alert alert-block alert-warning">
>>>
<b>TASK 1</b> — Using the <code>tree</code> object opened above:
<ol>
<li>Print the transverse momenta (<code>photon_pt</code>) for the first 5 events.</li>
<li>How many events does the first data file contain?  Print the answer.</li>
<li>Access the <code>photon_eta</code> branch and print the first event's two photon &eta; values.</li>
<li>Plot the two photon's in the first event in &eta;, &phi; coordinates and make the size of each photon marker proportional to the photon energy.</li>
</ol>
</div>

<a id='task2'></a>
## 3. Task 2 — Define the cut functions

The analysis requires five cuts.  Each cut returns a **boolean mask** (awkward
array) that is `True` for events that pass the cut and `False` for events that
should be removed.

| Cut | Branch(es) | Requirement |
|-----|-------------|-------------|
| Reconstruction quality | `photon_isTightID` | both photons `True` |
| Transverse momentum | `photon_pt` | leading $p_T>50$ GeV, sub-leading $p_T>30$ GeV |
| Calorimeter isolation | `photon_ptcone20`, `photon_pt` | $pTcone20/pT < 0.055$ for both |
| $\eta$ transition | `photon_eta` | exclude $1.37 < \|\eta\| < 1.52$ |
| $p_T/m$ isolation | `photon_pt`, invariant mass | $p_T/m_{\gamma\gamma} > 0.35$ for both |

<div class="alert alert-block alert-warning">
>>>
<b>TASK 2</b> — Implement the five cut functions below.  Each must accept
awkward arrays (per-event lists of 2 photons) and return a 1-D boolean mask.
The first cut is done for you as an example.
</div>

In [ ]:
# EXAMPLE: Photon reconstruction quality
def cut_photon_reconstruction(photon_isTightID):
    """Keep events where BOTH photons pass Tight ID."""
    return (photon_isTightID[:, 0] == True) & (photon_isTightID[:, 1] == True)

<a id='task3'></a>
## 4. Task 3 — Apply cuts to a single batch

Before running over all data, let's apply the cuts to a **single batch** from
the first file so we can verify each cut works.

<div class="alert alert-block alert-warning">
>>>
<b>TASK 3</b> — In the cell below:
<ol>
<li>Read one batch of <code>variables</code> from the tree using
<code>tree.iterate(..., entry_stop=500)</code>.</li>
<li>Apply the cuts <b>in order</b>: reconstruction → pT → isolation → eta → mass
calculation → mass-isolation.</li>
<li>Print how many events remain after <b>each</b> cut.</li>
</ol>
</div>

In [ ]:
# >>> TASK 3 — SOLUTION <<<
batch = tree.iterate(variables, library="ak", entry_stop=500)
data = next(batch)  # one batch from the iterator
n_start = len(data)
print(f"Events before cuts: {n_start}")


<a id='task4'></a>
## 5. Task 4 — Cut-flow table

A **cut-flow table** shows how many events survive after each successive cut.
This is an important diagnostics in any particle-physics analysis.

<div class="alert alert-block alert-warning">
>>>
<b>TASK 4</b> — Process the <b>entire first data file</b> and record the number
of events remaining after each cut.  Store the counts in a list called
<code>cut_counts</code> and print a formatted table.
</div>

In [ ]:
# >>> TASK 4 — SOLUTION <<<
cut_names = [
    "All events",
    "Photon reconstruction (TightID)",
    "Photon pT (50/30 GeV)",
    "Calorimeter isolation",
    "Eta transition exclusion",
    "Null mass removal",
    "pT/m relative isolation",
]

cut_counts = [0, 0, 0, 0, 0, 0, 0]

for data in tree.iterate(variables, library="ak"):
    # before any cuts
    n0 = len(data)
    cut_counts[0] += n0

    # this is my first cut:
    data = data[cut_photon_reconstruction(data['photon_isTightID'])]
    cut_counts[1] += len(data)

    # now over to you...
    data = data[]
    cut_counts[2] += len(data)

    #... all the way to cut 6 


<a id='task5'></a>
## 6. Task 5 — Invariant mass of a single event

The di-photon invariant mass is computed from the 4-momenta of the two photons:

$$m_{\gamma\gamma} = \sqrt{E_{\text{tot}}^2 - \mathbf{p}_{\text{tot}}\cdot\mathbf{p}_{\text{tot}}}$$

using natural units ($c=1$).

<div class="alert alert-block alert-warning">
>>>
<b>TASK 5</b> — For the first event in the first data file, calculate the
invariant mass <b>step by step</b> (try not to use the <code>calc_mass</code>
helper).  Print the photon 4-vectors and the resulting mass.
</div>

In [ ]:
# >>> TASK 5 — SOLUTION <<<
first_event = tree.arrays(library="ak")[:1]

# Extract individual photon 4-vector components for the first event
# photons are ordered by descending pT: [0] = leading, [1] = sub-leading
pt0  = first_event['photon_pt'][0, 0]    # transverse momentum of photon 0
pt1  = first_event['photon_pt'][0, 1]    # transverse momentum of photon 1
eta0 = first_event['photon_eta'][0, 0]   # pseudorapidity of photon 0
eta1 = first_event['photon_eta'][0, 1]   # pseudorapidity of photon 1
phi0 = first_event['photon_phi'][0, 0]   # azimuthal angle of photon 0
phi1 = first_event['photon_phi'][0, 1]   # azimuthal angle of photon 1
e0   = first_event['photon_e'][0, 0]      # energy of photon 0
e1   = first_event['photon_e'][0, 1]      # energy of photon 1


<a id='task6'></a>
## 7. Task 6 — Histogram the data after all cuts

Now we process the **full dataset** (all files), apply all cuts, and collect the
invariant masses.  We then histogram the result.

<div class="alert alert-block alert-warning">
>>>
<b>TASK 6</b> — Complete the processing loop.  The skeleton is provided; you
must insert the correct cut calls in the correct order.  After processing, create
a histogram of the invariant mass in the range 100–160 GeV with 1 GeV bins.
</div>

In [ ]:
# >>> TASK 6 — SOLUTION <<<
sample_data = []

for afile in files_list:
    print(f'Processing {files_list.index(afile)+1}/{len(files_list)}: {afile.split("/")[-1]}')
    tree = uproot.open(afile + ":analysis")
    numevents = tree.num_entries

    for data in tree.iterate(variables, library="ak",
                             entry_stop=int(numevents * fraction)):
        # Apply cuts in order
        
        # create new column called mass
        data['mass'] = 000


print("Done processing all data files.")

# Concatenate all masses into a single array
all_data = ak.concatenate(sample_data)
print(f"Total selected events: {len(all_data)}")


In [ ]:
# Histogram configuration
xmin, xmax = 100, 160    # GeV
step_size  = 1           # GeV

bin_edges   = np.arange(start=xmin, stop=xmax + step_size, step=step_size)
bin_centres = np.arange(start=xmin + step_size/2,
                         stop=xmax + step_size/2, step=step_size)

# Histogram the data
data_x, _ = np.histogram(ak.to_numpy(all_data), bins=bin_edges)
# Statistical errors: sqrt(N), protect against empty bins
data_x_errors = np.where(data_x > 0, np.sqrt(data_x), 1e-10)

print(f"Events in histogram (100-160 GeV): {data_x.sum()}")


### Plot the mass spectrum

This produces the main data histogram (black points).

In [ ]:
main_axes = plt.gca()
main_axes.errorbar(x=bin_centres, y=data_x, yerr=data_x_errors,
                    fmt='ko', label='Data', markersize=4)
main_axes.set_xlim(left=xmin, right=xmax)
main_axes.xaxis.set_minor_locator(AutoMinorLocator())
main_axes.tick_params(which='both', direction='in', top=True, right=True)
main_axes.set_xlabel(r'di-photon invariant mass $\mathrm{m_{\gamma\gamma}}$ [GeV]',
                      fontsize=13, x=1, horizontalalignment='right')
main_axes.set_ylabel('Events / '+str(step_size)+' GeV', y=1, horizontalalignment='right')
main_axes.set_ylim(bottom=0, top=np.amax(data_x)*1.6)
main_axes.yaxis.set_minor_locator(AutoMinorLocator())
main_axes.legend(frameon=False)
plt.show()


<a id='task6b'></a>
## 8. (Optional) Task 6b — Plotting the mass spectrum

Before proceeding, it could be interesting to look at the invariant mass distribution in different data taking years and periods. In a plot, compare data from 2015 to 2016 and different periods within each. Do you notice any differences?

<div class="alert alert-block alert-warning">
>>>
<b>TASK 6b</b> — Plot the data from different years and periods separately on the same axes with the same x-axis label as above. 
</div>

<a id='task7'></a>
## 8. Task 7 — Background-only fit

Before looking for a signal we need a model for the **background**.  The
di-photon mass spectrum is smoothly falling, so a 4th-order polynomial is a
good description:

$$B(m) = c_0 + c_1 m + c_2 m^2 + c_3 m^3 + c_4 m^4$$

<div class="alert alert-block alert-warning">
>>>
<b>TASK 7</b> — Fit the data with a <b>4th-order polynomial only</b> (no
Gaussian).  Store the fit result in <code>out_bkg</code> and the fitted
parameters in a dict.  Print the fitted polynomial coefficients.
</div>

In [ ]:
# >>> TASK 7 — SOLUTION <<<
# Background-only model
poly_mod = PolynomialModel(4)
pars_bkg = poly_mod.guess(data_x, x=bin_centres,
                           c0=data_x.max(), c1=0, c2=0, c3=0, c4=0)

out_bkg = poly_mod.fit(data_x, pars_bkg, x=bin_centres,
                        weights=1/data_x_errors)

# Extract coefficients
pdict_bkg = out_bkg.params.valuesdict()
print("Fitted background coefficients:")
for k in ['c0','c1','c2','c3','c4']:
    print(f"  {k} = {pdict_bkg[k]:.6g}")

# Evaluate background on bin centres
background_only = (pdict_bkg['c0']
                    + pdict_bkg['c1']*bin_centres
                    + pdict_bkg['c2']*bin_centres**2
                    + pdict_bkg['c3']*bin_centres**3
                    + pdict_bkg['c4']*bin_centres**4)

print(f"\nBackground-only reduced chi2 = {out_bkg.redchi:.2f}")
print("(Large reduced chi2 indicates structure not captured by background — that's the signal!)")


<a id='task8'></a>
## 9. Task 8 — Signal+background fit

Now add a Gaussian signal model on top of the polynomial background.  The
Gaussian describes the Higgs mass peak broadened by the detector's photon
energy resolution.

$$S(m) = \frac{A}{\sigma\sqrt{2\pi}}\exp\!\left(-\frac{(m-\mu)^2}{2\sigma^2}\right)$$

<div class="alert alert-block alert-warning">
>>>
<b>TASK 8</b> — Create a combined <b>polynomial + Gaussian</b> model, fit it to
the data, and extract the fitted Gaussian mean ($\mu$), width ($\sigma$), and
amplitude ($A$).  Store the fit result in <code>out</code>.
</div>

<a id='task9'></a>
## 10. Task 9 — Residuals and reduced $\chi^2$

A **residual plot** (Data − Background) is the standard way to visualise the
signal component.  The reduced $\chi^2$ tells us how well the fit describes the
data.

$$\chi^2 = \sum_i \frac{(N_i - f_i)^2}{\sigma_i^2}, \qquad
\chi^2_{\text{red}} = \frac{\chi^2}{N_{\text{bins}} - N_{\text{params}}}$$

<div class="alert alert-block alert-warning">
>>>
<b>TASK 9</b> —
<ol>
<li>Compute the residuals <code>data_x - background</code> and plot them as
error points.</li>
<li>Overlay the fitted signal component <code>out.best_fit - background</code>.</li>
<li>Compute the reduced $\chi^2$ from the fit result and print it.</li>
</ol>
</div>

In [ ]:
# >>> TASK 9 — SOLUTION <<<
# Residuals: data minus background-only component
residuals = data_x - background
residual_errors = data_x_errors

# Signal component from the fit
signal_component = out.best_fit - background

# --- Main plot: data + total fit + background ---
plt.axes([0.1, 0.3, 0.85, 0.65])
main_axes = plt.gca()

main_axes.errorbar(x=bin_centres, y=data_x, yerr=data_x_errors,
                   fmt='ko', label='Data', markersize=4)
main_axes.plot(bin_centres, out.best_fit, '-r',
                label='Sig+Bkg Fit ($m_H=125$ GeV)')
main_axes.plot(bin_centres, background, '--r',
               label='Bkg (4th order poly)')

main_axes.set_xlim(left=xmin, right=xmax)
main_axes.xaxis.set_minor_locator(AutoMinorLocator())
main_axes.tick_params(which='both', direction='in', top=True, right=True,
                       labelbottom=False)
main_axes.set_ylabel('Events / '+str(step_size)+' GeV',
                      horizontalalignment='right')
main_axes.set_ylim(bottom=0, top=np.amax(data_x)*1.5)
main_axes.yaxis.set_minor_locator(AutoMinorLocator())
main_axes.yaxis.get_major_ticks()[0].set_visible(False)

plt.text(0.2, 0.92, 'ATLAS Open Data', transform=main_axes.transAxes, fontsize=13)
plt.text(0.2, 0.86, 'for education', transform=main_axes.transAxes,
         style='italic', fontsize=8)
plt.text(0.2, 0.80,
         r'$\sqrt{s}$=13 TeV, $\int$L dt = '+str(luminosity*fraction)+' fb$^{-1}$',
         transform=main_axes.transAxes)
plt.text(0.2, 0.74, r'$H \rightarrow \gamma\gamma$', transform=main_axes.transAxes)

main_axes.legend(frameon=False, loc='lower left')

# --- Sub plot: residuals ---
plt.axes([0.1, 0.1, 0.85, 0.2])
sub_axes = plt.gca()
sub_axes.yaxis.set_major_locator(MaxNLocator(nbins='auto', symmetric=True))

# Plot residuals
sub_axes.errorbar(x=bin_centres, y=residuals, yerr=residual_errors,
                   fmt='ko', markersize=4, label='Data − Bkg')
# Plot fitted signal component
sub_axes.plot(bin_centres, signal_component, '-r', label='Fitted signal')

sub_axes.set_xlim(left=xmin, right=xmax)
sub_axes.xaxis.set_minor_locator(AutoMinorLocator())
sub_axes.set_xlabel(r'Di-photon invariant mass $\mathrm{m_{\gamma\gamma}}$ [GeV]',
                     x=1, horizontalalignment='right', fontsize=13)
sub_axes.tick_params(which='both', direction='in', top=True, right=True)
sub_axes.yaxis.set_minor_locator(AutoMinorLocator())
sub_axes.set_ylabel('Events − Bkg')
sub_axes.legend(frameon=False, loc='upper right', fontsize=8)

main_axes.yaxis.set_label_coords(-0.09, 1)
sub_axes.yaxis.set_label_coords(-0.09, 0.5)

plt.show()

# --- Reduced chi^2 ---
chi2      = out.chisqr
ndof       = out.nfree
red_chi2   = out.redchi
print(f"Chi2           = {chi2:.1f}")
print(f"NDF            = {ndof}")
print(f"Reduced chi2   = {red_chi2:.2f}")
print(f"A reduced chi2 close to 1 means the model describes the data well.")


<a id='task10'></a>
## 11. Task 10 — Signal yield and excess estimate

We can estimate the number of signal events in the Higgs peak by comparing
the data to the background-only fit in a **signal window** around the fitted
mean.

<div class="alert alert-block alert-warning">
>>>
<b>TASK 10</b> —
<ol>
<li>Define a signal window of $\mu \pm 2\sigma$ around the fitted Gaussian
mean.</li>
<li>Count the data events in this window.</li>
<li>Integrate the background-only fit over the same window.</li>
<li>Compute the excess: <code>N_signal = N_data − N_background</code>.</li>
<li>Estimate a simple "significance" as
$S / \sqrt{B}$ where $S$ is the excess and $B$ is the background.</li>
<li>Print all these numbers.</li>
</ol>
</div>

<a id='task11'></a>
## 12. Task 11 — Sideband cross-check

A **sideband cross-check** compares the data and the background model in
regions **away** from the signal peak.  If the background model is good, the
data in the sidebands should agree with the background prediction.

<div class="alert alert-block alert-warning">
>>>
<b>TASK 11</b> —
<ol>
<li>Define two sideband regions: 105–120 GeV (low) and 130–155 GeV (high).</li>
<li>Count data events and background events in each sideband.</li>
<li>Compute the ratio <code>Data / Background</code> in each sideband.  A ratio
close to 1 means the background model is working well.</li>
<li>Print the results.</li>
</ol>
</div>

In [ ]:
# >>> TASK 11 — SOLUTION <<<
# Sideband regions (avoid the 120-130 GeV signal window)
lo_side = (bin_centres >= 105) & (bin_centres < 120)
hi_side = (bin_centres > 130) & (bin_centres <= 155)

# Data vs background in each sideband
n_data_lo = np.sum(data_x[lo_side])
n_bkg_lo  = np.sum(background[lo_side])

n_data_hi = np.sum(data_x[hi_side])
n_bkg_hi  = np.sum(background[hi_side])

ratio_lo = n_data_lo / n_bkg_lo if n_bkg_lo > 1e-10 else float('nan')
ratio_hi = n_data_hi / n_bkg_hi if n_bkg_hi > 1e-10 else float('nan')

print("--- Sideband cross-check ---")
print(f"Low sideband (105-120 GeV):  Data={n_data_lo}, Bkg={n_bkg_lo:.1f}, "
      f"Data/Bkg={ratio_lo:.3f}")
print(f"High sideband (130-155 GeV): Data={n_data_hi}, Bkg={n_bkg_hi:.1f}, "
      f"Data/Bkg={ratio_hi:.3f}")
print()
print("If the ratios are close to 1, the background model is reliable.")
print("Any large deviation would suggest the polynomial order is not sufficient.")


<a id='task12'></a>
## 13. Task 12 — Limit-setting reasoning

In a real analysis, we ask: *"If there were **no** Higgs boson, how likely
would we be to see an excess this large?"*

We won't do heavy statistics here, but we can reason about the ingredients and
estimate an "upper limit" on the signal yield using a simple **counting
experiment**.

### Simple counting experiment

In the signal window $[\mu - 2\sigma, \mu + 2\sigma]$:
* $N_{\text{data}}$ — observed data events
* $N_{\text{bkg}}$ — expected background (from the fit)
* $S_{\text{obs}} = N_{\text{data}} - N_{\text{bkg}}$ — observed excess

If there were no Higgs, the data would fluctuate around $N_{\text{bkg}}$ with
uncertainty $\sqrt{N_{\text{bkg}}}$.  An **approximate 95% confidence-level
upper limit** on the signal yield can be estimated as:

$$S_{\text{limit}}^{95\%} \approx S_{\text{obs}} + 1.64\,\sqrt{N_{\text{bkg}}}$$

*(The factor 1.64 corresponds to the one-sided 95% quantile of a Gaussian.)*

<div class="alert alert-block alert-warning">
>>>
<b>TASK 12</b> —
<ol>
<li>Using the numbers from Task 10, compute $S_{\text{obs}}$ and
$S_{\text{limit}}^{95\%}$.</li>
<li>Print both values and comment on whether the observed excess exceeds the
expected limit (i.e. whether we would claim a "discovery-like" observation).</li>
</ol>
</div>

In [ ]:
# >>> TASK 12 — SOLUTION <<<
# Re-use the signal-window quantities from Task 10
N_data   = n_data_window
N_bkg    = n_bkg_window
S_obs    = N_data - N_bkg

# Approximate 95% CL upper limit on the signal yield
S_limit_95 = S_obs + 1.64 * np.sqrt(max(N_bkg, 1e-10))

print("=== Simple counting-experiment limit ===")
print(f"Signal window:       {window_lo:.1f} – {window_hi:.1f} GeV")
print(f"Observed data (N):   {N_data}")
print(f"Background (B):      {N_bkg:.1f}")
print(f"Observed excess (S):  {S_obs:.1f} events")
print(f"Stat. uncertainty on B (sqrt(B)): {np.sqrt(max(N_bkg,1e-10)):.1f}")
print()
print(f"Approximate 95% CL upper limit on signal: {S_limit_95:.1f} events")
print()
if S_obs > 1.64 * np.sqrt(max(N_bkg, 1e-10)):
    print("The observed excess exceeds the 95% CL expectation → "
          "evidence for a signal (Higgs!).")
else:
    print("The observed excess is below the 95% CL expectation → "
          "no significant signal claimed.")


<a id='task13'></a>
## 14. Task 13 — Optional: vary a cut and study the effect

A good analyst always checks how robust the result is against cut variations.

<div class="alert alert-block alert-warning">
>>>
<b>TASK 13 (optional)</b> — Re-run the analysis but change the leading-photon
$p_T$ cut from 50 GeV to <b>35 GeV</b>.  Observe how the number of selected
events changes and whether the Higgs peak becomes more or less visible.
</div>

---

## Summary

In this notebook you have:

1. **Explored** the ATLAS Open Data TTree structure.
2. **Implemented** five photon selection cuts.
3. **Built a cut-flow table** showing the effect of each cut.
4. **Computed** the di-photon invariant mass.
5. **Histogrammed** the full dataset after all cuts.
6. **Fitted** the background with a 4th-order polynomial.
7. **Fitted** the signal+background with a Gaussian + polynomial.
8. **Plotted residuals** and computed the reduced $\chi^2$.
9. **Estimated** the signal yield and a simple significance.
10. **Cross-checked** the background model in sidebands.
11. **Estimated** an approximate 95% CL upper limit on the signal yield.

The Higgs boson should be clearly visible as a peak around 125 GeV — you have
rediscovered it!

---

### Going further
* Try different polynomial orders (2nd, 3rd) for the background and compare the reduced $\chi^2$.
* Try different initial guesses for the Gaussian parameters and see if the fit converges to the same values.
* Compute the signal yield in a narrower window ($\mu \pm 1\sigma$) and compare.
* Process all the data (set `fraction = 1.0`) and compare the significance.

<div class="alert alert-block alert-info">
We welcome your feedback on this notebook or any of our other materials! Please
<a href="https://forms.gle/zKBqS1opAHHemv9U7">fill out this survey</a> to let us
know how we're doing, and you can enter a raffle to win some
<a href="https://atlas-secretariat.web.cern.ch/merchandise">ATLAS merchandise</a>!
</div>
